# Complexity-03 — More time, more problems

**Navigation**: [Index](README.md) | [02 — Verify or find](Complexity-02-P-NP-Reduction.ipynb) | [03b — Hartmanis–Stearns](Complexity-03b-HartmanisStearns-TimeHierarchy.ipynb) | [03c — Navigable zoo](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) | [Français →](Complexity-03-TimeHierarchy.ipynb)

> **Audience: Licence.** The historical demonstration and its multi-tape simulation are kept in the 03b deep dive.

## What this notebook assumes

- [01 — Counting steps](Complexity-01-StepCounting.ipynb): step size and budget;
- [02 — Verify or find](Complexity-02-P-NP-Reduction.ipynb): decision problem and algorithm.

**Objectives**: distinguish inclusion from strict separation of classes, understand a finite diagonal and explain why a table is not a proof of hierarchy. **Estimated duration**: 35 minutes.

**The theorem in one sentence.** The time hierarchy theorem (Hartmanis–Stearns, 1965) states that, for reasonable budgets, having strictly more time allows one to decide strictly more problems: some tasks genuinely require more steps, and no fast algorithm can simulate them. This notebook illustrates the **mechanisms** of that proof — nested budgets, diagonalization, bounded simulation — on finite tables; it does not claim to prove it, as every reading reminds.

## 1. Nested budgets

If a program answers in n steps, it also answers within a budget of n² steps. This shows an **inclusion** of budgets, not a separation of problem classes: two different programs on the same problem do not prove that no fast program can solve it. Let us compare the budgets over five sizes.

The inclusion is trivial: a program that halts within n steps also halts within n² steps. The difficulty lies elsewhere — showing that a problem requires *more* than n² steps means ruling out **all** fast programs, not just the one at hand. That gap between "I have an example" and "I have exhausted the possibilities" is precisely what this notebook digs into.

In [1]:
tailles = (2, 4, 8, 16, 32)
print(f"{'n':>3} | {'n²':>5} | {'2ⁿ':>12}")
for n in tailles:
    print(f"{n:>3} | {n * n:>5} | {2 ** n:>12}")
assert all(n * n <= 2 ** n for n in tailles)


  n |    n² |           2ⁿ
  2 |     4 |            4
  4 |    16 |           16
  8 |    64 |          256
 16 |   256 |        65536
 32 |  1024 |   4294967296


**Reading the result.** The costs are ordered for the sizes shown: at n = 32, the n² budget is worth 1,024 steps against 4,294,967,296 for 2ⁿ — the gap widens at every size. To conclude that one class has a problem absent from the other, one would have to rule out **all** algorithms within the smaller budget. The numerical table does not do that.

## 2. The diagonalization idea

Let us enumerate a few candidate machines. At row i, construction D chooses the opposite of machine number i's answer on input i. D therefore differs from every listed machine on at least one input. The list below is only a **finite** experiment: it does not represent all possible machines.

Diagonalization reverses the direction of the question: instead of looking for a fast algorithm, one builds a problem D that contradicts each candidate machine on its own ground — input i. Against a finite list, this works; against the infinity of machines, one would need a universal construction and bound, which section 4 discusses.

In [2]:
machines = (
    lambda i: i % 2 == 0,
    lambda i: i < 2,
    lambda i: True,
    lambda i: i % 3 == 0,
)
diagonale = {i: not machine(i) for i, machine in enumerate(machines)}
print(f"{'i':>2} | {'M_i(i)':>6} | {'D(i)':>5} | différents")
for i, machine in enumerate(machines):
    print(f"{i:>2} | {str(machine(i)):>6} | {str(diagonale[i]):>5} | {diagonale[i] != machine(i)}")
assert all(diagonale[i] != machine(i) for i, machine in enumerate(machines))


 i | M_i(i) |  D(i) | différents
 0 |   True | False | True
 1 |   True | False | True
 2 |   True | False | True
 3 |   True | False | True


**Reading the result.** The four rows display `True` in the "different" column: D(i) is the opposite of M_i(i) on each of the listed inputs (0, 1, 2 and 3). Infinitely many machines remain unlisted; this table therefore proves no separation of classes.

## 3. The simulation budget

A machine that does not answer before the limit cannot block the diagonal constructor indefinitely. Here the costs are **declared** by a finite model: they are not steps measured by a universal simulator. An unobserved answer is not a false verdict.

In the real proof, this limit is handled by a **universal clock**: each candidate machine is simulated step by step and stopped beyond the budget. Paying for both the machine and its simulator is precisely the overhead that makes the hierarchy hold — and which the finite model here does not capture.

In [3]:
candidates = ((True, 1), (False, 3), (True, 12), (False, 40))
def budget(i):
    return 2 * i + 4

print(f"{'i':>2} | {'pas':>3} | {'budget':>6} | {'observé':>7} | {'D(i)':>5}")
for i, (verdict, pas) in enumerate(candidates):
    observe = verdict if pas <= budget(i) else None
    diagonal = not observe if observe is not None else True
    print(f"{i:>2} | {pas:>3} | {budget(i):>6} | {str(observe):>7} | {str(diagonal):>5}")


 i | pas | budget | observé |  D(i)
 0 |   1 |      4 |    True | False
 1 |   3 |      6 |   False |  True
 2 |  12 |      8 |    None |  True
 3 |  40 |     10 |    None |  True


**Reading the result.** `None` means "not observed within the budget", never "false": on rows 2 and 3, the declared costs (12 and 40) exceed the budget (8 and 10), the simulation stops and the answer is not observed. A machine exceeding this budget on one input does not satisfy the uniform bound on **all** inputs, but the table does not characterize any entire class.

## 4. The theorem, without an experimental shortcut

The time hierarchy theorem establishes, under technical assumptions, that there exist decidable problems solvable with a sufficiently larger budget but impossible with the smaller one. One must enumerate all admissible machines, build a clock and pay the cost of universal simulation. Finite tables accomplish none of these proofs. [03b — Hartmanis–Stearns](Complexity-03b-HartmanisStearns-TimeHierarchy.ipynb) details the historical model. This result does not settle P versus NP.

Concretely, the separation is obtained by building a problem D that simulates each candidate machine M_i on its input i, answers the opposite, and does so within a budget only slightly larger than M_i's — hence a strictly larger class as the budget grows. The price is known: enumerate all machines, equip them with a universal clock, and simulate each one while paying for both the machine and its simulator.

## Exercises

The three exercises extend the demonstrations on the same finite objects. They do not ask you to prove the hierarchy — beyond the reach of a table — but to **feel** what each table shows and what it leaves open, by manipulating budgets, the diagonal and the simulation bound.

### Exercise 1 — Inclusion or separation?

Compare n² and n³ for n = 2, 4, 8, 16. Explain why these numbers do not prove that a problem becomes solvable only with the second budget.

- **Hint:** a table of numbers rules out no other algorithm.
- **Step 1:** produce the four pairs.
- **Step 2:** distinguish inclusion from separation.

In [4]:
comparaison_budgets = None  # TODO etudiant : couples (n², n³)
print('Exercice a completer : inclusion et separation')


Exercice a completer : inclusion et separation


### Exercise 2 — Another diagonal

Add a fifth function to `machines`, then rebuild D and explain why this finite table is still not a general proof.

- **Hint:** evaluate the new function on its own index.
- **Step 1:** add a machine.
- **Step 2:** recompute the five oppositions.

In [5]:
nouvelle_diagonale = None  # TODO etudiant : cinquieme machine et nouvelle diagonale
print('Exercice a completer : nouvelle diagonale')


Exercice a completer : nouvelle diagonale


### Exercise 3 — Overrunning the budget

Find the rows whose declared cost exceeds `budget(i)`. Propose a budget that covers all four costs and explain why this does not build a bound for all machines.

- **Hint:** compare the two numerical columns.
- **Step 1:** produce a new budget.
- **Step 2:** state the limit of this finite model.

In [6]:
budget_etendu = None  # TODO etudiant : couvrir les quatre couts
print('Exercice a completer : budget etendu')


Exercice a completer : budget etendu


## Conclusion

| Move | Verified here | Not demonstrated here |
|---|---|---|
| Budgets | Numerical inclusion | Strict separation of classes |
| Diagonal | Four finite contradictions | All admissible machines |
| Simulation | Halt on overrun | Bounded universal clock and cost |

**To keep in mind.** Three moves, three scopes: budget inclusion is measured, diagonalisation is tested on a finite list, bounded simulation is observed on an overrun — but no finite table enumerates all the machines nor builds the universal clock. The time hierarchy is a theorem about the infinite of algorithms; this notebook shows its mechanisms on the finite, so that the proof in 03b has some ground to stand on.

The theorem asserts the existence of problems genuinely requiring more time, not merely of slow programs. The deep dive [03b](Complexity-03b-HartmanisStearns-TimeHierarchy.ipynb) works through the historical result.

**Reference**: Hartmanis, J. and Stearns, R. E. (1965), *On the computational complexity of algorithms*, Transactions of the American Mathematical Society 117, 285–306.

## Relativisation: what a finite diagonal cannot settle

The three tables of this notebook — nested budgets, finite diagonal, bounded simulation — illustrate the **mechanisms** of a separation proof. The actual theorem (Hartmanis–Stearns 1965) is unconditional: it separates classes of problems, not merely programs. But the technique that carries it — diagonalisation — has a limit known since 1975, and that limit deserves its place here because it says exactly what this kind of move can and cannot settle.

**The oracle theorem (Baker–Gill–Solovay 1975).** There exists a recursive oracle A such that `P^A = NP^A`, and a recursive (sparse) oracle B such that `P^B ≠ NP^B`. An oracle is a free black box added to both sides of the equation; the result produced by BGS shows that the question "P = NP?" has **both** answers depending on the oracle. Immediate consequence: any proof technique that *relativises* — that applies as it stands in the presence of an arbitrary oracle — is structurally neutral on P vs NP, in both directions. The authors say so in their introduction: ordinary diagonalisation is among the techniques one expects to relativise, and one sees no general simulation method that would escape the objection either. (Baker, Gill, Solovay, *Relativizations of the P NP question*, SIAM Journal on Computing 4(4), 1975, p. 431 — copy archived in the cluster's bibliographic repository.)

**What this changes for the diagonal of this notebook.** Mentally add an "oracle" column to the table of section 2: the finite contradiction (machines ≠ at the diagonal point) survives unchanged. That is the mark of a relativising argument — it can therefore conclude nothing about P vs NP, only about the finite objects it exhibits. This is not a flaw of the notebook: the Hartmanis–Stearns separation itself is proved; the lesson is that the diagonal move taught here is the right tool for time hierarchies and the wrong one for the P vs NP question.

**A measurement caveat.** Mehlhorn (quoted in §3 of BGS): the oracles X with `P^X = NP^X` form a *meagre* set — in a precise sense, "most" oracles separate. The categorical intuition therefore leans towards separation, and yet the question remains open: relativisation is not a sign of an answer, it is the proof that the standard methods supply none.

**Contemporary echo.** The doubt voiced by Wolfram in 2026 — resolving P vs NP by a finite proof from the standard axioms, "the results here make me doubt it" — is the empirical face of the same caution: where BGS proves a theorem *about proof techniques*, the empirical census of small Turing machines only offers an expectation. The two cite each other with their respective standing: proved (1975) and expected (2026).

## Key takeaways

1. **Budgets are nested, not equal.** At n = 32, n² is 1 024 steps and 2ⁿ is 4 294 967 296: more than six orders of magnitude for the same input size. It is that gap that makes "more time" capable of buying something — and that is what has to be proved, not assumed.
2. **The deferred diagonal is measured on four machines.** D(i) is the opposite of M_i(i) on all four rows (`different = True` everywhere): this is a **constructed** contradiction, not a theorem.
3. **Bounded simulation halts on overrun.** At i = 2 the simulation costs 12 steps against a budget of 8 and returns `None`; at i = 3, 40 steps against a budget of 10, also `None`. `None` means "not observed within the budget", never "false": confusing the two would look like a refutation.
4. **A finite table does not prove the hierarchy.** The conclusion says it in three columns: numerical inclusion verified, strict separation of classes not demonstrated, universal clock and bounded cost absent. [03b — Hartmanis–Stearns](Complexity-03b-HartmanisStearns-TimeHierarchy.ipynb) carries the real statement, and the relativisation section bounds the method.

## Check your understanding

1. At n = 32, n² is 1 024 and 2ⁿ is 4 294 967 296. What property must this comparison satisfy for "more time" to actually buy something?
2. On row i = 2, the simulation costs 12 steps against a budget of 8 and returns `None`. In terms of classes, what does a machine that overruns its budget correspond to?
3. The conclusion's table distinguishes "verified here" from "not demonstrated here". Which row falls in the second column, and why can it not be demonstrated with four machines?
4. The relativisation section states that there is an oracle for which P = NP and another for which P ≠ NP. What does that say about diagonalisation as a method?

<details><summary>Answers</summary>

1. The budgets must be **strictly nested**: each time class must strictly contain the previous one, so that no harder problem can be solved by the same machine. At n = 32 the gap between n² and 2ⁿ exceeds six orders of magnitude — the two budgets are so far apart that an n²-budget machine cannot do the work of a 2ⁿ-budget machine.
2. It does not belong to the class of that budget: a machine that overruns its budget is not in TIME(f) for the budget function at hand. That is why the verdict is `None` (not observed) and not `False` — confusing them would announce a refutation the machine never produced.
3. Strict separation: four machines and four sizes only show numerical inclusion, never that no machine could do better. The instrument that makes the statement provable is the one in [03b — Hartmanis–Stearns](Complexity-03b-HartmanisStearns-TimeHierarchy.ipynb): an exact step count and a budgeted simulation.
4. That diagonalisation cannot settle P against NP **in an absolute world**: its power depends on the oracle surrounding it, since the same technique yields both answers depending on the oracle. This is a limitation of the method, not a failure of the execution.

</details>

## Going further

- **Next notebook**: [Complexity-04 — Deciding without knowing what comes next](Complexity-04-OnlineAlgorithms-Python.ipynb) — the online algorithm: the same cost question, but the information arrives out of order.
- **In the same series**: [03b — Hartmanis–Stearns](Complexity-03b-HartmanisStearns-TimeHierarchy.ipynb) — the real theorem and its instrumentation step by step ; [03c — Navigable zoo](Complexity-03c-ComplexityZoo-Navigation-Python.ipynb) — the two oracle worlds, executed.
- **Neighbouring notebook from another series**: [CSP-1 — Fundamentals](../Search/Part2-CSP/CSP-1-Fundamentals.ipynb) — the search space and its measured cost, in another series.
- **Readings**: Baker, Gill and Solovay (1975), *Relativizations of the P=?NP Question*, SIAM J. Comput. 4-4 — the oracle theorem that bounds diagonalisation (copy archived in the shared gisement, folder `Bibliographie IA/Automata/`) ; Impagliazzo and Paturi (2001), *On the Complexity of k-SAT* (same folder).